# BERT Data Pipeline

## Objective

The objective of this notebook is to prepare the processed mental health dataset for BERT fine-tuning.

In this notebook, i will:

- Load the processed datasets
- Encode class labels
- Load the BERT tokenizer
- Tokenize the text
- Build PyTorch Dataset objects
- Create DataLoaders

The output of this notebook will be ready-to-use batches for BERT training.

## 1. Import Libraries

Import the required Python libraries for data loading, preprocessing, PyTorch, and Hugging Face Transformers.

In [1]:
from pathlib import Path

import random
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

## 2. Reproducibility

Set random seeds to make the experiments reproducible.

Using the same random seed helps produce the same results across different runs.

In [2]:
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Project Paths

I define reusable project paths.

Using Path objects improves code readability and portability.

In [3]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODEL_CACHE_DIR = PROJECT_ROOT / "models" / "huggingface_cache"

## 4. Load Processed Data

Load the processed training, validation, and test datasets that were created in Notebook 02.

In [4]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")

validation_df = pd.read_csv(PROCESSED_DATA_DIR / "validation.csv")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")

In [5]:
print(train_df.shape)

print(validation_df.shape)

print(test_df.shape)

(34254, 2)
(7340, 2)
(7341, 2)


In [6]:
display(train_df.head())

,text,status
0,@harbars managed to fix broken moss in 2 min a...,Normal
1,"It's 3 o'clock, the call to prayer for the mor...",Normal
2,@ShaileeMody thats a killing smile.. i m flatt...,Normal
3,How to stop waking up feeling suicidal?For the...,Suicidal
4,I know some people that want to live and turn ...,Depression


## 5. Label Encoding

Machine learning models cannot directly use text labels.

Therefore, each mental health category is converted into a numerical label.

The mapping will remain consistent across the training, validation, and test datasets.

In [7]:
label2id = {"Anxiety": 0, "Depression": 1, "Normal": 2, "Suicidal": 3}

In [8]:
train_df["label"] = train_df["status"].map(label2id)

validation_df["label"] = validation_df["status"].map(label2id)

test_df["label"] = test_df["status"].map(label2id)

In [9]:
train_df.head()

,text,status,label
0,@harbars managed to fix broken moss in 2 min a...,Normal,2
1,"It's 3 o'clock, the call to prayer for the mor...",Normal,2
2,@ShaileeMody thats a killing smile.. i m flatt...,Normal,2
3,How to stop waking up feeling suicidal?For the...,Suicidal,3
4,I know some people that want to live and turn ...,Depression,1


In [10]:
print(train_df["label"].value_counts().sort_index())

label
0     3727
1     9981
2    12706
3     7840
Name: count, dtype: int64


## 6. Load the BERT Tokenizer

The BERT tokenizer converts raw text into numerical representations that the model can process.

It performs several preprocessing steps automatically, including:

- Splitting text into subword tokens
- Converting tokens into token IDs
- Adding special tokens
- Creating attention masks

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased", cache_dir=MODEL_CACHE_DIR
)

In [12]:
print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 30522


In [13]:
sentence = "I feel very sad today."
encoded = tokenizer(sentence, return_tensors="pt")
print(encoded)

{'input_ids': tensor([[ 101, 1045, 2514, 2200, 6517, 2651, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


## 7. Create a Custom PyTorch Dataset

PyTorch models do not work directly with Pandas DataFrames.

Therefore, we create a custom Dataset class that converts each training example into the tensors required by BERT.

In [14]:
from torch.utils.data import Dataset

In [15]:
class MentalHealthDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=128):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

In [16]:
def __len__(self):
    return len(self.dataframe)

In [17]:
def __len__(self):
    return len(self.dataframe)

In [18]:
class MentalHealthDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = self.df.iloc[index]["text"]
        label = self.df.iloc[index]["label"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [19]:
train_dataset = MentalHealthDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=128,
)

print("Dataset size:", len(train_dataset))
print(train_dataset[0])

Dataset size: 34254
{'input_ids': tensor([  101,  1030,  5292, 28483,  2869,  3266,  2000,  8081,  3714, 10636,
         1999,  1016,  8117,  1998,  3828,  9703,  1012,  2092,  2589,  1001,
        11867,  2497, 14289,  2243,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,  

In [20]:
sample = train_dataset[0]

print(sample.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [21]:
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])

torch.Size([128])
torch.Size([128])
tensor(2)


## 8. Create DataLoaders

PyTorch uses DataLoader objects to load data efficiently during training.

The DataLoader creates mini-batches, shuffles the training data, and feeds batches to the model.

In [22]:
train_dataset = MentalHealthDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=128,
)

validation_dataset = MentalHealthDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=128,
)

test_dataset = MentalHealthDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=128,
)

In [23]:
BATCH_SIZE = 16

In [24]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [25]:
batch = next(iter(train_loader))

In [26]:
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])


In [27]:
print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


## 9. Validate the DataLoaders

Before model training, the DataLoaders are validated by inspecting one batch from each dataset.

This ensures that:

- the expected keys are present,
- tensor dimensions are correct,
- labels use the correct integer data type,
- and the batches are ready for BERT.

In [28]:
def inspect_batch(data_loader, split_name):
    batch = next(iter(data_loader))

    print(f"{split_name} batch")
    print("-" * 50)
    print("Keys:", batch.keys())
    print("Input IDs shape:", batch["input_ids"].shape)
    print("Attention mask shape:", batch["attention_mask"].shape)
    print("Labels shape:", batch["labels"].shape)
    print("Labels dtype:", batch["labels"].dtype)
    print()


inspect_batch(train_loader, "Training")
inspect_batch(validation_loader, "Validation")
inspect_batch(test_loader, "Test")

Training batch
--------------------------------------------------
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])
Labels dtype: torch.int64

Validation batch
--------------------------------------------------
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])
Labels dtype: torch.int64

Test batch
--------------------------------------------------
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])
Labels dtype: torch.int64



## 10. Number of Batches

The number of batches depends on the dataset size and batch size.

Each epoch processes all batches in the training DataLoader once.

In [29]:
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training samples: 34254
Validation samples: 7340
Test samples: 7341

Training batches: 2141
Validation batches: 459
Test batches: 459


## 11. Inspect a Tokenized Training Example

One example from the first training batch is decoded to verify that the token IDs correspond to the original text structure.

Special tokens and padding tokens are retained for inspection.

In [30]:
batch = next(iter(train_loader))

sample_input_ids = batch["input_ids"][0]
sample_attention_mask = batch["attention_mask"][0]
sample_label_id = batch["labels"][0].item()

decoded_tokens = tokenizer.convert_ids_to_tokens(sample_input_ids)

decoded_text = tokenizer.decode(sample_input_ids, skip_special_tokens=False)

print("Label ID:", sample_label_id)
# print("Label name:", id2label[sample_label_id])

print("\nAttention mask:")
print(sample_attention_mask)

print("\nTokens:")
print(decoded_tokens)

print("\nDecoded text:")
print(decoded_text)

Label ID: 3

Attention mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])

Tokens:
['[CLS]', 'we', 'cannot', 'even', 'act', 'surprised', 'either', '.', 'he', 'told', 'all', 'of', 'us', 'how', 'depressed', 'he', 'was', '.', 'the', 'last', 'time', 'i', 'saw', 'him', 'he', 'said', 'his', 'mind', 'was', 'going', 'to', 'dark', 'places', '.', 'the', 'way', 'he', 'said', 'it', 'though', 'was', 'so', 'ambiguous', '.', 'he', 'would', 'always', 'smile', 'and', 'laugh', 'after', '.', 'id', 'give', 'a', 'generic', 'response', 'like', 'oh', 'do', 'not', 'say', 'that', ',', 'or', 'some', 'bullshit', '.', 'when', 'i', '

## 12. Final Pipeline Validation

The following checks confirm that the complete data pipeline is ready for BERT fine-tuning.

In [31]:
train_batch = next(iter(train_loader))

assert set(train_batch.keys()) == {
    "input_ids",
    "attention_mask",
    "labels",
}

assert train_batch["input_ids"].ndim == 2
assert train_batch["attention_mask"].ndim == 2
assert train_batch["labels"].ndim == 1

assert train_batch["input_ids"].shape[0] == BATCH_SIZE
assert train_batch["input_ids"].shape[1] == 128

assert train_batch["input_ids"].shape == (train_batch["attention_mask"].shape)

assert train_batch["labels"].dtype == torch.long

assert train_batch["labels"].min().item() >= 0
assert train_batch["labels"].max().item() < len(label2id)

print("All pipeline validation checks passed.")  # heyyy

All pipeline validation checks passed.


## Key Takeaways

In this notebook, a complete BERT data pipeline was created.

The main steps included:

- loading the processed training, validation, and test datasets,
- encoding text labels as numerical class IDs,
- creating consistent `label2id` and `id2label` mappings,
- loading the `bert-base-uncased` tokenizer,
- examining input IDs, attention masks, token type IDs, and special tokens,
- creating a custom PyTorch `Dataset`,
- converting each example into BERT-compatible tensors,
- creating training, validation, and test `DataLoader` objects,
- organizing the data into mini-batches,
- and validating the final tensor shapes and data types.

The data pipeline is now ready for BERT fine-tuning.